# 12 — Validation, Stats, and Reporting

**Purpose:** Exercise TorchLens's self-verification and reporting surfaces: `tl.validate`,
`tl.report.explain`, `tl.tap`/`tl.record_span` observers, `summary()` at multiple levels,
`output_table`, and `tl.debug` power-user diagnostics.

**Surfaces covered:**
- [ ] `tl.validate(model, x, scope='forward')` — forward parity check
- [ ] `tl.validate(model, x, scope='saved')` — saved-out round-trip check
- [ ] `tl.validate(model, x, scope='backward', loss_fn=...)` — backward parity check
- [ ] `tl.report.explain(trace)` — human-readable model/capture narrative
- [ ] `tl.report.explain(trace, audience='auto')` — audience kwarg
- [ ] `tl.tap(site)` — `TapObserver`: attach + collect forward tensors
- [ ] `TapObserver.records` and `TapObserver.values()`
- [ ] `TapRecord` fields: `value`, `site_label`, `span_names`, `timestamp`, `direction`
- [ ] `tl.record_span(name)` — named span around a capture
- [ ] Span metadata dict fields
- [ ] `trace.summary()` — default and all named levels
- [ ] `trace.output_table()` — GAP callout (logits not retained without semantic decode)
- [ ] `tl.debug.bisect_nan(trace)` — locate first NaN/Inf in saved activations (§10)
- [ ] `tl.debug.hot_path(trace)` — source lines ranked by FLOP cost (§10)

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

## 2. `tl.validate` — forward scope

`tl.validate(model, x, scope='forward')` re-runs the model and checks that
TorchLens's recorded ops match the un-instrumented output.  Returns `True` if
all checks pass, `False` otherwise.  The validation system is a **tripwire**,
not a formality — a failure means the capture has a real bug.

In [ ]:
model, x = ZOO["tiny_mlp"]()

result_fwd = tl.validate(model, x, scope="forward")
print(f"validate scope='forward' : {result_fwd}  (True = all forward checks pass)")
print()

# More complex model
model2, x2 = ZOO["demo_model"]()
result_fwd2 = tl.validate(model2, x2, scope="forward")
print(f"validate demo_model fwd  : {result_fwd2}")

# CNN model
model3, x3 = ZOO["tiny_branch_cnn"]()
result_fwd3 = tl.validate(model3, x3, scope="forward")
print(f"validate tiny_branch_cnn : {result_fwd3}")

# BatchNorm model (has buffers)
model4, x4 = ZOO["batch_norm"]()
result_fwd4 = tl.validate(model4, x4, scope="forward")
print(f"validate batch_norm      : {result_fwd4}")

## 3. `tl.validate` — saved scope

Scope `'saved'` checks that saved output tensors round-trip faithfully
(shapes, dtypes, values).

In [ ]:
model, x = ZOO["tiny_mlp"]()
result_saved = tl.validate(model, x, scope="saved")
print(f"validate scope='saved'  : {result_saved}")

model5, x5 = ZOO["reused_relu_loop"]()
result_saved5 = tl.validate(model5, x5, scope="saved")
print(f"validate reused_relu_loop saved: {result_saved5}")

## 4. `tl.validate` — backward scope

Scope `'backward'` checks gradient fidelity.  Requires a `loss_fn` callable
that reduces the model output to a scalar.

In [ ]:
from _models import LinearReluModel

model_bwd = LinearReluModel()
x_bwd = torch.randn(1, 3, requires_grad=True)

# loss_fn must reduce model output to a scalar
result_bwd = tl.validate(
    model_bwd,
    x_bwd,
    scope="backward",
    loss_fn=lambda out: out.sum(),
)
print(f"validate scope='backward' : {result_bwd}")
print()

# MLP backward validation
model_mlp, x_mlp = ZOO["tiny_mlp"]()
x_mlp.requires_grad_(True)
result_bwd2 = tl.validate(
    model_mlp,
    x_mlp,
    scope="backward",
    loss_fn=lambda out: out.sum(),
)
print(f"validate tiny_mlp bwd     : {result_bwd2}")
print()
print("Note: tl.validate returns a plain bool. Verbose mode does not yet print to stdout;")
print("diagnostic output (if any) goes to the logger. A structured result object with")
print("per-check details would make this surface more debuggable.")

## 5. `tl.report.explain(trace)` — human-readable narrative

`tl.report.explain` returns a multi-section string covering model summary,
capture summary, anomalies, interventions, and notable patterns.  The `audience`
kwarg can adjust verbosity.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

report_text = tl.report.explain(trace)
print("=== tl.report.explain(trace) ===")
print(report_text)

In [ ]:
# More complex model for a richer report
model2, x2 = ZOO["demo_model"]()
trace2 = tl.trace(model2, x2)

report2 = tl.report.explain(trace2, audience="auto")
print("=== tl.report.explain(demo_model) ===")
print(report2)

## 6. `tl.tap` — attach a `TapObserver` to collect tensors

`tl.tap(site)` creates a `TapObserver` that records every activation at `site`
without modifying it.  Pass it as an action inside `tl.when(selector, observer)`,
then pass that to `intervene=` on the trace.  Collected records are in `.records`;
raw tensors via `.values()`.

In [ ]:
model, x = ZOO["tiny_mlp"]()

# Create a tap observer targeting all relu ops
relu_tap = tl.tap(tl.func("relu"))
print(f"TapObserver type    : {type(relu_tap).__name__}")
print(f"TapObserver repr    : {relu_tap}")
print()

# Wire it into a trace via tl.when(selector, action)
trace = tl.trace(
    model,
    x,
    intervene=tl.when(tl.func("relu"), relu_tap),
)

print(f"Records collected   : {len(relu_tap.records)}")
print(f"TapRecord type      : {type(relu_tap.records[0]).__name__}")
print()
rec = relu_tap.records[0]
print("TapRecord fields:")
print(f"  .value          : {rec.value}")
print(f"  .site_label     : {rec.site_label!r}")
print(f"  .span_names     : {rec.span_names}")
print(f"  .direction      : {rec.direction!r}")
print(f"  .grad_kind      : {rec.grad_kind!r}")
print()
print("TapObserver.values() (list of tensors):")
vals = relu_tap.values()
for i, v in enumerate(vals):
    print(f"  [{i}] shape={v.shape} mean={v.float().mean():.4f}")

In [ ]:
# Tap multiple sites on a model with a loop
model2, x2 = ZOO["reused_relu_loop"]()

loop_tap = tl.tap(tl.func("relu"))
trace2 = tl.trace(
    model2,
    x2,
    intervene=tl.when(tl.func("relu"), loop_tap),
)

print("ReusedReluLoop — relu called 4x in loop")
print(f"  Records collected: {len(loop_tap.records)}  (expect 4)")
print("  Activations evolve across iterations:")
for i, v in enumerate(loop_tap.values()):
    print(f"    iter {i}: mean={v.float().mean():.4f}  max={v.float().max():.4f}")

## 7. `tl.record_span` — named observer span

`tl.record_span(name)` is a context manager that annotates a block with a span
name.  Spans appear in `TapRecord.span_names` for records fired inside the block,
and yield a mutable metadata dict with timing fields.

In [ ]:
model, x = ZOO["tiny_mlp"]()
relu_tap2 = tl.tap(tl.func("relu"))

with tl.record_span("capture_window") as span_meta:
    trace3 = tl.trace(
        model,
        x,
        intervene=tl.when(tl.func("relu"), relu_tap2),
    )

print("Span metadata dict:")
for k, v in span_meta.items():
    print(f"  {k:10s}: {v}")
print()

if relu_tap2.records:
    rec2 = relu_tap2.records[0]
    print(f"TapRecord.span_names inside span: {rec2.span_names}")
    print("  (span name 'capture_window' is present)")
else:
    print("No records (unexpected — tap should fire on relu)")

print()

# Span duration
duration_ms = (span_meta["end"] - span_meta["start"]) * 1000
print(f"Span duration: {duration_ms:.1f} ms")
print()
print("Ergonomic note: span_meta is a plain dict; a named dataclass or NamedTuple")
print("would make field access cleaner (e.g. span.start vs span_meta['start']).")

## 8. `trace.summary()` — all named levels

`summary()` accepts a `level=` kwarg for domain-oriented views.  Available levels:
`'overview'` (default), `'graph'`, `'memory'`, `'control_flow'`, `'compute'`,
`'cost'`, `'waterfall'`, `'output'`.

In [ ]:
model, x = ZOO["demo_model"]()
trace_demo = tl.trace(model, x)

levels = ["overview", "graph", "memory", "control_flow", "compute", "cost", "waterfall", "output"]

for level in levels:
    try:
        s = trace_demo.summary(level=level)
        lines = s.strip().split("\n")
        print(f"--- level='{level}' ({len(lines)} lines) ---")
        # Print first 8 lines of the level-specific section (skip common preamble)
        section_start = next(
            (
                i
                for i, ln in enumerate(lines)
                if any(
                    kw in ln
                    for kw in [
                        "Model:",
                        "Graph Summary:",
                        "Memory Summary:",
                        "Control Flow:",
                        "Compute Summary:",
                        "Cost Summary:",
                        "Waterfall",
                        "Output Summary:",
                    ]
                )
            ),
            0,
        )
        for ln in lines[section_start : section_start + 12]:
            print(ln)
        print()
    except Exception as exc:
        print(f"level='{level}' ERROR: {exc}")
        print()

## 9. `trace.output_table()`

`output_table()` is designed for models with semantic output decoding (logit tables,
top-k predictions, etc.). It requires the trace to have been captured with
output-decoding configuration. Without it, the method raises `ValueError`.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

try:
    tbl = trace.output_table()
    print(tbl)
except ValueError as exc:
    print("⚠️ GAP — output_table() raised ValueError:")
    print(f"  {exc}")
    print()
    print("Expected: a table of model output values / top-k predictions.")
    print("Actual  : requires semantic output decoding (logit retention) at capture time.")
    print("Status  : output_table() is only useful for classification/LM models where")
    print("          tl.trace was called with output decoding enabled. The zoo models")
    print("          do not configure that; the surface exists but cannot be exercised here.")
    print("          See notebook 14 (HF-guarded) for the intended usage.")

## 10. `tl.debug` — power-user diagnostics

`tl.debug` is a submodule (not in `tl.__all__`) that provides diagnostic tools for
power users.  Two key functions: `bisect_nan` locates the first NaN/Inf in saved
activations, and `hot_path` finds the source lines accounting for the most FLOP cost.

These are imported as `tl.debug.bisect_nan` and `tl.debug.hot_path`.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

# --- tl.debug.bisect_nan ---
# Scans saved activations for NaN/Inf; returns a BisectNanResult.
print("=== tl.debug.bisect_nan(trace) ===")
print("type:", type(tl.debug.bisect_nan).__name__)

result_nan = tl.debug.bisect_nan(trace)
print("result type :", type(result_nan).__name__)
print("result repr :", result_nan)
print()
# Inspect fields
print("result.found       :", result_nan.found)  # False — no NaN in tiny_mlp
print("result.op          :", result_nan.op)  # None if not found
print("result.label       :", result_nan.label)
print("result.kind        :", result_nan.kind)
print("result.message     :", result_nan.message)
print()

# Confirm on a model that injects NaN
import torch as _torch
import torch.nn as _nn


class NaNModel(_nn.Module):
    def forward(self, x):
        out = _torch.log(x * -1)  # log of negative = NaN
        return out


model_nan = NaNModel()
x_nan = _torch.ones(1, 4)
try:
    trace_nan = tl.trace(model_nan, x_nan)
    result_nan2 = tl.debug.bisect_nan(trace_nan)
    print("=== bisect_nan on NaN-producing model ===")
    print("result.found  :", result_nan2.found)
    print("result.kind   :", result_nan2.kind)
    print("result.label  :", result_nan2.label)
    print("result.message:", result_nan2.message)
except Exception as exc:
    print(f"  NaN model trace raised {type(exc).__name__}: {exc}")

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

# --- tl.debug.hot_path ---
# Returns a DataFrame ranked by FLOP cost per source line.
print("=== tl.debug.hot_path(trace) ===")
print("type:", type(tl.debug.hot_path).__name__)

hp = tl.debug.hot_path(trace)
print("result type    :", type(hp).__name__)
print("result columns :", hp.columns.tolist())
print()

# Print the top rows — truncate source_file:line to avoid huge paths
import os as _os


def _truncate_path(row):
    path = str(row.get("source_file:line", ""))
    # Keep only last two path components + line number
    parts = path.split(_os.sep)
    if len(parts) >= 2:
        return _os.sep.join(parts[-2:])
    return path


hp_display = hp.copy()
hp_display["source"] = [_truncate_path(r) for _, r in hp_display.iterrows()]
hp_display = hp_display[["source", "op_count", "total_cost", "pct_total"]]

print("Top source lines by FLOP cost:")
print(hp_display.to_string(index=False))
print()
print("Note: tl.debug is NOT in tl.__all__; access as tl.debug.hot_path / tl.debug.bisect_nan.")

# hot_path also accepts 'memory' or 'duration' as the 'by' metric
hp_mem = tl.debug.hot_path(trace, by="memory")
print()
print(f"hot_path(trace, by='memory') — {len(hp_mem)} rows, metric: memory")

---

## ⚠️ GAPs / ergonomic smells

- **`tl.validate` returns a plain `bool`** — no structured result: no per-check breakdown, no failing layer names, no tolerance details. When it returns `False`, the user has no idea what failed or where. A `ValidationResult(passed=False, failures=[...])` would be far more debuggable.
- **`tl.validate` `verbose=True` does not print to stdout** — verified: stdout is empty with `verbose=True`. Diagnostic output goes somewhere else (logger?). A user who passes `verbose=True` expecting console output gets nothing; this is surprising.
- **`tl.report.explain` `audience=` kwarg is not yet differentiated** — passing different audience values (e.g. `'auto'`, `'novice'`, `'expert'`) appears to produce identical output on simple models. The surface exists but the differentiation is not visible here.
- **`tl.record_span` yields a plain `dict`** — `span_meta['start']` vs a typed dataclass. Not blocking but less ergonomic. `TapRecord` is a proper dataclass; span should match.
- **`tl.tap` is not a context manager** — it must be wired through `tl.when(selector, observer)` and passed to `intervene=`. This is correct but not immediately obvious from the function signature; the docstring should prominently show the `tl.when` pattern.
- **`trace.output_table()` not exercisable without semantic output decoding** — documented above as a GAP. The surface exists; exercising it requires HF/classifier models with output-decoding config.